In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

In [2]:
with open(r"C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\UI\BERT\processed_movies.pkl", 'rb') as file:
    df = pickle.load(file)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45379 entries, 0 to 45378
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                45379 non-null  int64  
 1   genres                45379 non-null  object 
 2   id                    45379 non-null  int64  
 3   original_language     45379 non-null  object 
 4   original_title        45379 non-null  object 
 5   overview              45379 non-null  object 
 6   popularity            45379 non-null  float64
 7   production_companies  45378 non-null  object 
 8   production_countries  45378 non-null  object 
 9   release_date          45294 non-null  object 
 10  revenue               45378 non-null  float64
 11  runtime               45379 non-null  float64
 12  spoken_languages      45378 non-null  object 
 13  status                45297 non-null  object 
 14  title                 45379 non-null  object 
 15  vote_average       

In [3]:
with open(r"C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\UI\BERT\movie_embeddings.pkl", 'rb') as file:
    embeddings = pickle.load(file)

embeddings

array([[-0.09277535, -0.03705722,  0.00831588, ...,  0.01836774,
         0.07707796,  0.02492109],
       [-0.06669601,  0.0596749 , -0.02984753, ..., -0.02898323,
        -0.09959951, -0.0405548 ],
       [-0.10162781, -0.04958753, -0.01619051, ..., -0.03369964,
        -0.01661375,  0.02921417],
       ...,
       [-0.04727595, -0.03389184, -0.04593802, ..., -0.00576479,
        -0.04783046, -0.07779828],
       [-0.0208091 ,  0.06657801, -0.07509368, ..., -0.06488372,
        -0.02205547, -0.07099294],
       [-0.0369518 , -0.01533157, -0.09306476, ..., -0.04310708,
         0.02229875, -0.09952784]], shape=(45379, 384), dtype=float32)

In [4]:
df.shape

(45379, 33)

In [5]:
embeddings.shape

(45379, 384)

In [6]:
cols = ['avg_rating', 'popularity', 'vote_count', 'year']

scaler = MinMaxScaler()

df[cols] = scaler.fit_transform(df[cols])

In [7]:
nn = NearestNeighbors(
    n_neighbors=30,
    metric='cosine',
    algorithm='brute'
)

nn.fit(embeddings)

,n_neighbors,30
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [8]:
def recommend(movie_name, top_n=10):
    movie_name = movie_name.strip()
    
    matches = df[df['title'].str.lower() == movie_name.lower()]

    if len(matches) == 0:
        print("Movie not found")
        return

    idx = matches.index[0]

    query_vector = embeddings[idx].reshape(1, -1)

    distances, indicies = nn.kneighbors(query_vector)

    query_genres = set(str(df.iloc[idx]['genres']).split())

    results = []

    for rank, i in enumerate(indicies[0]):

        if i == idx:
            continue

        sim = 1 - distances[0][rank]

        candidate_genres = set(str(df.iloc[i]['genres']).split())
        genre_score = len(query_genres.intersection(candidate_genres))
        genre_score = min(genre_score / 3, 1)

        year_diff = abs(df.iloc[i]['year'] - df.iloc[idx]['year'])
        year_score = max(0, 1 - year_diff / 10)

        final_score = (
            0.55 * sim +
            0.15 * genre_score +
            0.10 * df.iloc[i]['avg_rating'] +
            0.10 * df.iloc[i]['popularity'] +
            0.05 * df.iloc[i]['vote_count'] +
            0.05 * year_score
        )

        results.append((i, final_score))

    results = sorted(results, key=lambda x: x[1], reverse=True)

    output = []

    for i, score in results[:top_n]:
        output.append({
            'Title': df.iloc[i]['title'],
            'Score': round(score, 3),
            'Rating': round(df.iloc[i]['avg_rating'], 3),
            'Popularity': round(df.iloc[i]['popularity'], 3),
            'Year': round(df.iloc[i]['year'], 3)
        })

    return pd.DataFrame(output)

In [9]:
recommend("Toy Story")

,Title,Score,Rating,Popularity,Year
0,Toy Story 2,0.731,0.763,0.032,0.990
1,Toy Story 3,0.727,0.778,0.031,0.995
2,The Toy,0.614,0.547,0.011,0.981
3,Babes in Toyland,0.608,0.711,0.014,0.957
4,Toys,0.605,0.550,0.011,0.986
5,The Toy,0.598,0.749,0.004,0.978
6,Super Buddies,0.593,0.580,0.006,0.997
7,Tin Toy,0.591,0.664,0.011,0.984
8,Toy Story of Terror!,0.590,0.680,0.001,0.997
9,Partysaurus Rex,0.587,0.673,0.011,0.996


In [10]:
recommend("Heat")

,Title,Score,Rating,Popularity,Year
0,Hot Pursuit,0.703,0.558,0.013,0.998
1,Going in Style,0.698,0.644,0.028,0.999
2,Carlito's Way,0.678,0.740,0.016,0.987
3,15 Minutes,0.676,0.597,0.022,0.991
4,Spotlight,0.674,0.815,0.027,0.998
5,Masterminds,0.668,0.545,0.017,0.998
6,Mulholland Drive,0.665,0.764,0.033,0.991
7,Changeling,0.663,0.739,0.021,0.994
8,American Gangster,0.661,0.762,0.018,0.994
9,Fury,0.661,0.752,0.005,0.958


In [11]:
recommend("Jumanji")

,Title,Score,Rating,Popularity,Year
0,The Game,0.573,0.769,0.027,0.989
1,Toy Story 2,0.556,0.763,0.032,0.990
2,BMX Bandits,0.541,0.567,0.014,0.982
3,Monster House,0.539,0.658,0.028,0.993
4,Bridge to Terabithia,0.538,0.686,0.015,0.994
5,Nerve,0.537,0.693,0.027,0.998
6,Diary of a Wimpy Kid: Dog Days,0.535,0.624,0.019,0.996
7,Big,0.534,0.722,0.017,0.984
8,Pete's Dragon,0.534,0.622,0.022,0.998
9,Lemony Snicket's A Series of Unfortunate Events,0.533,0.665,0.023,0.992


In [12]:
recommend("Titanic")

,Title,Score,Rating,Popularity,Year
0,Titanic,0.680,0.500,0.006,0.988
1,Beauty and the Beast,0.651,0.717,0.525,0.999
2,Avatar,0.650,0.733,0.338,0.995
3,Titanic,0.646,0.667,0.024,0.967
4,Wreck-It Ralph,0.638,0.750,0.025,0.996
5,Star Wars: The Force Awakens,0.623,0.767,0.058,0.998
6,Django Unchained,0.617,0.802,0.036,0.996
7,Maleficent,0.606,0.677,0.036,0.997
8,Titanic: The Final Word with James Cameron,0.605,0.750,0.001,0.996
9,Pirates of the Caribbean: Dead Men Tell No Tales,0.604,0.674,0.244,0.999


In [13]:
#pickle.dump(nn, open(r"C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\UI\BERT\nn_model.pkl", 'wb'))
print('Model Saved')

Model Saved


In [14]:
#pickle.dump(df, open(r"C:\Users\HP\OneDrive\Desktop\AIML\Practice\Datasets\Movie recommendation\UI\BERT\processed_scaled_movies.pkl", 'wb'))
print("Scaled Dataset Saved")

Scaled Dataset Saved


In [18]:
df["popularity"].describe()

count    45379.000000
mean         0.005338
std          0.010977
min          0.000000
25%          0.000705
50%          0.002060
75%          0.006722
max          1.000000
Name: popularity, dtype: float64

In [4]:
df['cast_clean'].head()

0    Tom Hanks Tim Allen Don Rickles Jim Varney Wal...
1    Robin Williams Jonathan Hyde Kirsten Dunst Bra...
2    Walter Matthau Jack Lemmon Ann-Margret Sophia ...
3    Whitney Houston Angela Bassett Loretta Devine ...
4    Steve Martin Diane Keaton Martin Short Kimberl...
Name: cast_clean, dtype: object